# 01 – Exploratory Data Analysis

**Dataset:** NYC Vehicle Collisions  
**Source:** https://www.kaggle.com/datasets/nypd/vehicle-collisions

This notebook provides an initial exploration of the raw dataset:
- Data loading and shape inspection
- Missing-value audit
- Temporal distribution of collisions
- Spatial distribution across NYC boroughs
- Quick visualisation with matplotlib / seaborn

> **Before running:** make sure the raw CSV is present.  
> Download it with: `uv run python data/download_data.py`

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid")

print(f"pandas  {pd.__version__}")
print(f"torch   {torch.__version__}")

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_FILE = PROJECT_ROOT / "data" / "raw" / "vehicle_collisions.csv"

assert DATA_FILE.exists(), (
    f"Data file not found: {DATA_FILE}\n"
    "Run `uv run python data/download_data.py` to download it."
)

df = pd.read_csv(DATA_FILE, low_memory=False)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
df.head()

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.3)))
missing.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("% missing")
ax.set_title("Missing values by column")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "figures" / "missing_values.png", dpi=150)
plt.show()

In [ ]:
# ── Temporal distribution ─────────────────────────────────────────────────────
date_col = next(
    (c for c in df.columns if "date" in c.lower()), None
)

if date_col:
    df["_date"] = pd.to_datetime(df[date_col], errors="coerce")
    monthly = df.set_index("_date").resample("ME").size()

    fig, ax = plt.subplots(figsize=(12, 4))
    monthly.plot(ax=ax, color="tomato")
    ax.set_title("Monthly collision count")
    ax.set_xlabel("Date")
    ax.set_ylabel("Collisions")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "reports" / "figures" / "monthly_collisions.png", dpi=150)
    plt.show()
else:
    print("No date column detected – skipping temporal plot.")

In [ ]:
# ── Spatial distribution ──────────────────────────────────────────────────────
if "BOROUGH" in df.columns:
    borough_counts = (
        df["BOROUGH"].value_counts().reset_index()
    )
    borough_counts.columns = ["Borough", "Count"]

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=borough_counts, x="Borough", y="Count", palette="viridis", ax=ax)
    ax.set_title("Collisions per Borough")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "reports" / "figures" / "borough_distribution.png", dpi=150)
    plt.show()
else:
    print("No BOROUGH column detected – skipping spatial plot.")